# Notebook 2 — Five-Minute AIS Interpolation Grid with Interactive Validation Map

Stage ini mempertahankan kontrak ilmiah baseline: grid lima menit, bracket observasi sebelum–sesudah, tanpa carry-forward atau extrapolation, interpolasi linier/circular, pembentukan state operasional, audit, peta validasi, dan keluaran CSV. Logika transformasi berada pada `src/mfar_stage12.py` agar dapat diuji tanpa mengubah hasil.


In [ ]:
#@title Inisialisasi repository dan jalur MFAR
from pathlib import Path
from datetime import datetime, timezone
import os, subprocess, sys

REPOSITORY_URL = "https://github.com/yanto-mashardi/MFAR_Modular_Colab_Pipeline.git"
DEFAULT_REF = "refactor/prompt1-stage01-02"
REPOSITORY_REF = os.environ.get("MFAR_GIT_REF", DEFAULT_REF)

def locate_repository() -> Path:
    candidates = []
    override = os.environ.get("MFAR_CODE_ROOT", "").strip()
    if override:
        candidates.append(Path(override))
    candidates.extend([
        Path.cwd(),
        Path.cwd().parent,
        Path("/content/MFAR_Modular_Colab_Pipeline"),
        Path("/content/drive/MyDrive/MFAR_Modular_Colab_Pipeline"),
        Path("/content/drive/MyDrive/MFAR_Modular_Colab_Pipeline/MFAR_Modular_Colab_Pipeline"),
    ])
    for candidate in candidates:
        if (candidate / "src" / "mfar_paths.py").is_file():
            return candidate.resolve()

    clone_target = Path("/content/MFAR_Modular_Colab_Pipeline")
    if "google.colab" in sys.modules or Path("/content").is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPOSITORY_REF,
             REPOSITORY_URL, str(clone_target)],
            check=True,
        )
        if (clone_target / "src" / "mfar_paths.py").is_file():
            return clone_target.resolve()
    raise FileNotFoundError(
        "Repository MFAR tidak ditemukan. Tetapkan MFAR_CODE_ROOT atau clone repository terlebih dahulu."
    )

CODE_ROOT = locate_repository()
os.environ["MFAR_CODE_ROOT"] = str(CODE_ROOT)
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

from src.mfar_paths import (
    AIS_RAW_PATH, VEHICLE_ARRIVAL_PATH, CONFIG_DIR,
    STAGE_01_DIR, STAGE_02_DIR,
)

_MFAR_STARTED_AT = datetime.now(timezone.utc)
print("Repository:", CODE_ROOT)
print("Input/output root:", STAGE_01_DIR.parent.parent)


In [ ]:
#@title Jalankan Stage 02: grid 5 menit, interpolasi, state, audit, dan ekspor
from src.mfar_stage12 import run_stage2

NOTEBOOK_NAME = "02_Time_Grid_and_State_Preparation.ipynb"
result = run_stage2(
    input_ais=STAGE_01_DIR / "01_ais_clean.csv",
    profile_file=CONFIG_DIR / "vessel_profiles.csv",
    berth_file=CONFIG_DIR / "terminal_berths.csv",
    stage_dir=STAGE_02_DIR,
    notebook_name=NOTEBOOK_NAME,
    started_at=_MFAR_STARTED_AT,
    grid_interval_min=5,
    max_bracket_gap_min=20,
)

print("Grid diterima:", len(result.output_grid))
print("Grid ditolak:", len(result.rejected_grid))
print("Audit interpolasi gagal:", int(result.audit_checks["failed_rows"].sum()))
print("Audit state gagal:", int(result.operational_audit["failed_rows"].sum()))


In [ ]:
#@title Laporan penyelesaian Stage 02
print("Stage 02 selesai.")
for path in result.saved_files:
    print("-", path.name)
